# Realized-volatility benchmark — ModernTCN vs HAR-RVRuns **both** model families over **all three horizons** (h = 1 daily, 5 weekly, 22 monthly)on `data/realized_volatility.csv`, and puts the results in one table.Both predict the same object, so the numbers are comparable row for row:$$Y_t^{(h)} = \ln\!\left(\tfrac{1}{h}\sum_{k=1}^{h} RV_{t+k}\right)$$one number per origin — the log of the **arithmetic** mean RV over the next *h* days, with the logoutside the sum. Inputs are $\ln(RV)$ on both sides.**Splits** come from `data_provider/splits.py`, which both families read:| | train | validation | test ||---|---|---|---|| ModernTCN | 2010-01-01 – 2021-12-31 | 2022-01-01 – 2023-12-31 | 2024-01-01 – 2025-04-07 || HAR-RV | 2010-01-01 – 2023-12-31 (train + val) | — | 2024-01-01 – 2025-04-07 |HAR-RV is OLS with nothing to tune, so it folds validation into training. The **test window isidentical**, giving 328 / 324 / 307 forecast origins at h = 1 / 5 / 22 for both.**Runtime.** HAR-RV takes seconds. ModernTCN is 3 horizons × 5 seeds; on a Colab GPU expectroughly 10–20 minutes total, longer on CPU. Lower `EPOCHS` or `SEEDS` in the config cell toshorten it. `Runtime → Change runtime type → GPU` is worth setting, but not required.

## 1. Setup

In [ ]:
import os, sys, subprocess

REPO   = "https://github.com/Mr0022/ModernTCNt.git"
BRANCH = "claude/moderntcn-aggregation-log-ptj8ao"
ROOT   = "/content/ModernTCNt"

if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO, ROOT], check=True)
else:
    print("Repository already cloned; pulling the latest commit.")
    subprocess.run(["git", "-C", ROOT, "pull", "--ff-only"], check=False)

# os.chdir, not %cd: it moves the Python process, so the ! cells below inherit it.
os.chdir(os.path.join(ROOT, "ModernTCN-Long-term-forecasting"))
print("working directory:", os.getcwd())

# Colab ships torch, pandas, numpy, sklearn, matplotlib and scipy. statsmodels is
# what HAR-RV needs for its OLS and HAC covariance; install only if it is missing.
try:
    import statsmodels  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "statsmodels"], check=True)

import torch, pandas as pd
print("torch", torch.__version__, "| GPU:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")
print("rows in data/realized_volatility.csv:", len(pd.read_csv("data/realized_volatility.csv")))

## 2. Configuration`CFG` holds the Bayesian-optimisation result for each horizon — one search per horizon, so thethree configurations differ. `SEEDS` is how many seeds each horizon is trained over: `--itr` is aseed count, and seed *i* is `2021 + i`, so 5 seeds means 2021–2025. Every seed is trained fromscratch and metrics are averaged at the end, because one seed is a draw rather than a result.

In [ ]:
HORIZONS = [1, 5, 22]     # h = daily, weekly, monthly
SEEDS    = 5              # --itr: seeds 2021 .. 2021+SEEDS-1
EPOCHS   = 100            # early stopping usually stops well before this
PATIENCE = 10
DES      = "Colab"        # tags the output filenames

# Best hyperparameters per horizon (Bayesian optimisation).
CFG = {
    1:  dict(seq_len=22, patch_size=32, patch_stride=8, ffn_ratio=3, large_size=51,
             small_size=3, num_blocks=3, dropout=0.4155, head_dropout=0.2993,
             learning_rate=0.00550, batch_size=128),
    5:  dict(seq_len=35, patch_size=32, patch_stride=2, ffn_ratio=3, large_size=13,
             small_size=5, num_blocks=3, dropout=0.0751, head_dropout=0.3690,
             learning_rate=0.00431, batch_size=128),
    22: dict(seq_len=35, patch_size=4,  patch_stride=8, ffn_ratio=2, large_size=31,
             small_size=5, num_blocks=1, dropout=0.2332, head_dropout=0.3228,
             learning_rate=0.00779, batch_size=256),
}

import pandas as pd
pd.DataFrame(CFG).rename_axis("hyperparameter").rename_axis("horizon", axis=1)

## 3. HAR-RV — Corsi (2009), log scaleOne invocation fits all three horizons. `--log` runs the whole pipeline on the ln(RV) scale;`--asset forex` selects the calendar above. OLS has no hyperparameters, so there is nothing toseed and nothing to average.

In [ ]:
!python -u HAR-RV_RUN.PY --data data/realized_volatility.csv --log --asset forex --outdir results_har

## 4. ModernTCN — aggregated target, log scaleOne run per horizon, each over `SEEDS` seeds. `--pred_len` is the **horizon h**, not a pathlength: the head emits a single number. Per-seed MSE / MAE / QLIKE are printed as the run goes,followed by the mean and standard deviation across seeds.

In [ ]:
for h in HORIZONS:
    c = CFG[h]
    cmd = (
        "python -u run.py"
        " --is_training 1 --model ModernTCN --model_id RV_h{h}"
        " --data RV --root_path ./data/ --data_path realized_volatility.csv"
        " --features S --target RV --asset forex --freq d --enc_in 1"
        " --seq_len {seq_len} --label_len 0 --pred_len {h}"
        " --patch_size {patch_size} --patch_stride {patch_stride} --ffn_ratio {ffn_ratio}"
        " --large_size {large_size} --small_size {small_size} --num_blocks {num_blocks}"
        " --dims 32 32 32 32 --dw_dims 32 32 32 32"
        " --dropout {dropout} --head_dropout {head_dropout}"
        " --learning_rate {learning_rate} --batch_size {batch_size}"
        " --revin 1 --random_seed 2021 --itr {seeds}"
        " --train_epochs {epochs} --patience {patience} --des {des}"
        " --lradj type3 --use_multi_scale False --small_kernel_merged False"
        " --num_workers 2"
    ).format(h=h, seeds=SEEDS, epochs=EPOCHS, patience=PATIENCE, des=DES, **c)

    print("\n" + "#" * 78)
    print(f"#  ModernTCN  h={h}  --  {SEEDS} seed(s), seq_len={c['seq_len']}")
    print("#" * 78)
    # !{cmd} runs the string in a subshell and streams its output into the cell.
    !{cmd}

## 5. Results`MSE` and `MAE` are in **ln(RV)** units — the scale both families are fitted on. `QLIKE`(Patton, 2011) needs variances, so it is evaluated after `exp()` with the lognormal Jensencorrection, `E[RV|F] = exp(E[ln RV|F] + σ²/2)`; `MSE_RV` and `MAE_RV` are the same forecasts onthat back-transformed scale. ModernTCN columns are the mean over seeds, ± the standard deviation.Lower is better throughout.

In [ ]:
import pandas as pd, numpy as np
from IPython.display import display

har = pd.read_csv("results_har/har_rv_log_all_metrics.csv")
har = har[har["split"] == "test"].set_index("horizon")

METRICS = ["MSE", "MAE", "QLIKE", "MSE_RV", "MAE_RV"]
rows = []
for h in HORIZONS:
    tcn = pd.read_csv(f"results/RV_h{h}_{DES}_seed_metrics.csv")
    tcn["seed"] = tcn["seed"].astype(str)
    mean = tcn[tcn.seed == "mean"].iloc[0]
    std  = tcn[tcn.seed == "std"].iloc[0]
    n_seeds = (~tcn.seed.isin(["mean", "std"])).sum()
    for m in METRICS:
        rows.append({"horizon": h, "metric": m,
                     "HAR-RV": float(har.loc[h, m]),
                     "ModernTCN": float(mean[m]),
                     "ModernTCN_std": float(std[m]),
                     "seeds": int(n_seeds)})

res = pd.DataFrame(rows)
res["better"] = np.where(res["ModernTCN"] < res["HAR-RV"], "ModernTCN", "HAR-RV")
res["improvement_%"] = 100 * (res["HAR-RV"] - res["ModernTCN"]) / res["HAR-RV"]

pretty = res.assign(**{
    "ModernTCN (mean ± sd)": res.apply(
        lambda r: f"{r['ModernTCN']:.6f} ± {r['ModernTCN_std']:.6f}", axis=1),
    "HAR-RV ": res["HAR-RV"].map("{:.6f}".format),
    "Δ%": res["improvement_%"].map("{:+.1f}".format),
})[["horizon", "metric", "HAR-RV ", "ModernTCN (mean ± sd)", "Δ%", "better"]]

print(f"Test window 2024-01-01 .. 2025-04-07, identical rows for both models.")
print(f"ModernTCN averaged over {res.seeds.iloc[0]} seeds. Δ% > 0 means ModernTCN is lower.\n")
display(pretty.set_index(["horizon", "metric"]))
res.to_csv("results/comparison_moderntcn_vs_har.csv", index=False)
print("\nSaved: results/comparison_moderntcn_vs_har.csv")

### Per-seed detailThe spread across seeds is what says whether a gap between the two families is real or noise.

In [ ]:
from IPython.display import display

for h in HORIZONS:
    d = pd.read_csv(f"results/RV_h{h}_{DES}_seed_metrics.csv")
    print(f"\n--- h = {h} " + "-" * 60)
    display(d[["seed", "MSE", "MAE", "QLIKE", "MSE_RV", "MAE_RV"]]
            .set_index("seed").round(6))

### LaTeX tableBooktabs, ready to paste. Numbers are plain — swap in your `\raa{}{}` macro if the documentneeds the RTL decimal form.

In [ ]:
LABELS = {"MSE": r"MSE $[\ln]$", "MAE": r"MAE $[\ln]$", "QLIKE": r"QLIKE $[RV]$",
          "MSE_RV": r"MSE$_{RV}$", "MAE_RV": r"MAE$_{RV}$"}

lines = [r"\begin{table}[htbp]", r"\centering",
         r"\caption{Out-of-sample forecast losses, test window 2024-01-01--2025-04-07. "
         r"ModernTCN is the mean over %d seeds. Lower is better.}" % res.seeds.iloc[0],
         r"\label{tab:rv-comparison}",
         r"\begin{tabular}{l" + "cc" * len(HORIZONS) + "}", r"\toprule",
         r"& " + " & ".join(r"\multicolumn{2}{c}{$h=%d$}" % h for h in HORIZONS) + r" \\",
         r"\textbf{Metric} & " + " & ".join(["HAR-RV & ModernTCN"] * len(HORIZONS)) + r" \\",
         r"\midrule"]
for m in METRICS:
    cells_ = []
    for h in HORIZONS:
        r = res[(res.horizon == h) & (res.metric == m)].iloc[0]
        cells_ += [f"{r['HAR-RV']:.4f}", f"{r['ModernTCN']:.4f}"]
    lines.append(LABELS[m] + " & " + " & ".join(cells_) + r" \\")
lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]

latex = "\n".join(lines)
print(latex)
with open("results/comparison_table.tex", "w") as f:
    f.write(latex)
print("\nSaved: results/comparison_table.tex")

## 6. HAR-RV figures`HAR-RV_RUN.PY` writes five publication figures into `results_har/`. They describe the HAR fititself — forecast paths, loss comparison, coefficient paths across horizons, residual ACFs andscatter grids.

In [ ]:
from IPython.display import Image, display
import glob, os

for f in sorted(glob.glob("results_har/*.png")):
    print("\n" + os.path.basename(f))
    display(Image(filename=f, width=900))

## 7. Where the outputs are| path | what ||---|---|| `results/RV_h{h}_Colab_seed_metrics.csv` | ModernTCN, one row per seed plus mean and std || `results/<setting>/rv_forecasts.csv` | per-origin forecast, dated by the first day of the target window || `results/<setting>/rv_metrics.csv` | one run's losses, HAR-RV's column names || `results/comparison_moderntcn_vs_har.csv` | the table above || `results/comparison_table.tex` | the same, as LaTeX || `results_har/har_rv_log_all_metrics.csv` | HAR-RV losses, all horizons || `results_har/har_rv_log_h{h}_params.csv` | HAR-RV coefficients, std errors, HAC t-stats |Colab discards `/content` when the runtime ends. To keep the results, download them or mountDrive and copy `results/` and `results_har/` across.